In [1]:
import math
import random

def euclidean_distance(a: tuple, b: tuple) -> float:
    return math.sqrt((a[0] - b[0])**2 + (a[1] - b[1])**2)

def tour_length(tour, dist):
    n = len(tour)
    return sum(dist[tour[i]][tour[(i + 1) % n]] for i in range(n))

def load_tsp(filename: str):
    coords = []
    with open(filename) as f:
        for line in f:
            if line.strip() == 'NODE_COORD_SECTION':
                break
        for line in f:
            if line.strip() == 'EOF':
                break
            parts = line.split()
            x, y = float(parts[1]), float(parts[2])
            coords.append((x, y))
    n = len(coords)
    dists = [[euclidean_distance(coords[i], coords[j])
              for j in range(n)]
             for i in range(n)]
    return coords, dists

In [ ]:
def ant_system_tsp(filename, iterations=10, num_ants=None,
                   alpha=1, beta=1, evaporation_rate=0.5, Q=100):

    coords, distances = load_tsp(filename)
    num_cities = len(coords)

    if num_ants is None:
        num_ants = num_cities

    initial_pheromone = 1.0 / num_cities
    pheromone_matrix = [[initial_pheromone for _ in range(num_cities)]
                        for _ in range(num_cities)]

    visibility_matrix = [[0.0 if city_i == city_j else 1.0 / distances[city_i][city_j]
                          for city_j in range(num_cities)]
                         for city_i in range(num_cities)]

    global_best_tour = None
    global_best_length = float('inf')

    for iteration in range(iterations):

        iteration_tours = []
        iteration_tour_lengths = []

        for ant in range(num_ants):

            visited_cities = [random.randint(0, num_cities - 1)]
            current_city = visited_cities[0]

            for _ in range(num_cities - 1):
                unvisited_cities = [city for city in range(num_cities)
                                    if city not in visited_cities]

                transition_weights = [
                    (pheromone_matrix[current_city][next_city] ** alpha)
                    * (visibility_matrix[current_city][next_city] ** beta)
                    for next_city in unvisited_cities
                ]
                total_weight = sum(transition_weights)

                if total_weight == 0:
                    next_city = random.choice(unvisited_cities)
                else:
                    selection_probs = [w / total_weight for w in transition_weights]
                    spin = random.random()
                    cumulative_prob = 0
                    next_city = unvisited_cities[-1]
                    for idx, candidate_city in enumerate(unvisited_cities):
                        cumulative_prob += selection_probs[idx]
                        if spin <= cumulative_prob:
                            next_city = candidate_city
                            break

                visited_cities.append(next_city)
                current_city = next_city

            ant_tour_length = tour_length(visited_cities, distances)
            iteration_tours.append(visited_cities)
            iteration_tour_lengths.append(ant_tour_length)

            if ant_tour_length < global_best_length:
                global_best_length = ant_tour_length
                global_best_tour = visited_cities.copy()

        pheromone_deposits = [[0.0 for _ in range(num_cities)]
                              for _ in range(num_cities)]

        for ant in range(num_ants):
            deposit_amount = Q / iteration_tour_lengths[ant]
            for step in range(num_cities):
                city_from = iteration_tours[ant][step]
                city_to = iteration_tours[ant][(step + 1) % num_cities]
                pheromone_deposits[city_from][city_to] += deposit_amount
                pheromone_deposits[city_to][city_from] += deposit_amount  

        for city_i in range(num_cities):
            for city_j in range(num_cities):
                pheromone_matrix[city_i][city_j] = (
                    (1 - evaporation_rate) * pheromone_matrix[city_i][city_j]
                    + pheromone_deposits[city_i][city_j]
                )
                pheromone_matrix[city_i][city_j] = max(
                    pheromone_matrix[city_i][city_j], 1e-10
                )
        if iteration % 10 == 0 :
            print(f"Iteration {iteration}: best = {global_best_length}")

    return global_best_tour, global_best_length

tour, length = ant_system_tsp('../Lab3Assigment3/berlin52.tsp',iterations=500)
print(f"Global best after 10 iterations: {length}")

Iteration 0: best = 19808.6741132753
Iteration 10: best = 10170.688562627704
Iteration 20: best = 9057.565429439355
Iteration 30: best = 9057.565429439355
Iteration 40: best = 8626.97642752364
Iteration 50: best = 8626.97642752364
Iteration 60: best = 8626.97642752364
Iteration 70: best = 8319.197229928874
Iteration 80: best = 8319.197229928874
Iteration 90: best = 8319.197229928874
Iteration 100: best = 8319.197229928874
Iteration 110: best = 8319.197229928874
Iteration 120: best = 8319.197229928874
Iteration 130: best = 8319.197229928874
Iteration 140: best = 8319.197229928874
Iteration 150: best = 8319.197229928874
Iteration 160: best = 8319.197229928874
Iteration 170: best = 8319.197229928874
Iteration 180: best = 8319.197229928874
Iteration 190: best = 8281.881157043466
Iteration 200: best = 8281.881157043466
Iteration 210: best = 8281.881157043466
Iteration 220: best = 8281.881157043466
Iteration 230: best = 8281.881157043466
Iteration 240: best = 8281.881157043466
Iteration 250: